# PlantVillage: Does ImageNet pretraining help with limited labels?
Minerva University Deep Learning progress demo. Compare random and ImageNet-pretrained ResNet-18 using the **same** leaf-grouped 10% subset, validation set, preprocessing, Adam learning rate, batch size and five-epoch budget. Fully fine-tune both models. The published test set is reserved for final evaluation.

**Status:** this notebook has not yet been run for plant classification. Outputs are intentionally empty. Synthetic software tests are not experimental results.

After the repository has been pushed with your approval, open this notebook through **Colab → File → Open notebook → GitHub**, enter `Ifethecoder/plant-disease-classifier`, and choose `notebooks/progress_demo.ipynb`. Select **Runtime → Change runtime type → GPU**. Before publication you can upload the notebook and the project ZIP to Colab, extract it, and set `PROJECT_DIR` below to the extracted repository root. Allow space for the 2.18 GB source archive plus selected color images and dependencies. Downloaded data stays outside Git.

In [ ]:
# Bootstrap from GitHub after the approved push, or point to an uploaded project.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Ifethecoder/plant-disease-classifier.git"
PROJECT_DIR = Path.cwd() if (Path.cwd() / "src/dataset.py").exists() else Path.cwd() / "plant-disease-classifier"
if not (PROJECT_DIR / "src/dataset.py").exists():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
# If torch was imported before this cell, restart the runtime after installation.
# Then run from this cell again. Installation should happen before any torch imports.

## GPU, reproducibility and output storage
Save artifacts directly to Google Drive so checkpoints and histories survive runtime disconnections. Change `PERSIST_TO_DRIVE` to `False` to use local Colab storage, then download the ZIP at the end before disconnecting. Use a new run directory each time; existing results are never silently overwritten.

In [ ]:
import tempfile
from datetime import datetime, timezone

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
CACHE_DIR = Path(tempfile.gettempdir()) / "plant-village-cache"
os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Choose a GPU in Runtime → Change runtime type, then rerun.")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)

PERSIST_TO_DRIVE = True
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = Path("/content/drive/MyDrive/plant-disease-classifier/results")
else:
    RESULTS_ROOT = PROJECT_DIR / "results"
RUN_DIR = RESULTS_ROOT / datetime.now(timezone.utc).strftime("demo_%Y%m%d_%H%M%S_utc")
print("Results:", RUN_DIR)

## Inspect schema and splits before training
Only metadata is loaded here. The upstream repository contains a legacy loading script; the project uses Hugging Face downloads and `datasets.Dataset` image records built from the pinned published color manifests, without remote-code execution.

A metadata audit found 227 shared leaf IDs across the published train/test manifests under the upstream mapping rules. Quarantine matching **training** records (290 images at this revision); leave every test record intact. Use leaf IDs for the new class-aware train/validation partition. Filename-derived fallback groups are conservative and cannot guarantee unknown physical-leaf identities.

In [ ]:
from src.dataset import load_metadata, make_splits

metadata = load_metadata(CACHE_DIR / "hub")
print("Pinned revision:", metadata["revision"])
print("Schema:", metadata["upstream_schema"])
print("Published splits:", metadata["published_counts"])
print("Classes:", len(metadata["classes"]))
split_preview = make_splits(metadata, fraction=0.10, seed=42, val_fraction=0.20)
print("Development counts:", split_preview["counts"])
print("Actual image fraction:", split_preview["actual_image_fraction"])
assert len(metadata["classes"]) == 38
assert split_preview["counts"]["test_reserved"] == 10709

## Train the matched comparison
Both models use Adam (`lr=0.001`, no weight decay), batch size 32, 224×224 RGB images with ImageNet normalization, cross-entropy, the same shuffle seed, and five epochs. There are no optimization-setting differences. Both backbones are trainable. Select each best checkpoint by validation accuracy, breaking ties by validation loss. The lower training count reflects the 10% **eligible training leaf-group** subset after validation/quarantine; exact image fractions vary with group sizes.

The runner downloads the source archive automatically, extracts only selected development color images into a temporary cache, initializes both ResNet-18 variants and runs them sequentially. This is the expensive cell. Never report its results until it completes successfully. A runtime disconnection requires a new run directory; checkpoint resume is not implemented.

In [ ]:
from dataclasses import asdict
from src.train import TrainConfig
from src.experiment import run_comparison

config = TrainConfig(fraction=0.10, seed=42, val_fraction=0.20,
                     epochs=5, batch_size=32, learning_rate=0.001,
                     image_size=224, num_workers=2, device="cuda",
                     freeze_backbone=False)
print(asdict(config))
comparison = run_comparison(config, RUN_DIR, CACHE_DIR)

## Validation results and evidence
These are development validation results from checkpoints selected on this same set. Accuracy is the proportion correct; macro F1 averages class F1 equally across all 38 classes. Treat a single seed and short training budget as preliminary evidence. The untouched test set is not evaluated by this notebook.

In [ ]:
from IPython.display import display, Markdown, Image as DisplayImage

display(Markdown((RUN_DIR / "comparison.md").read_text()))
for name in ("random", "pretrained"):
    display(Markdown(f"### {name.title()} ResNet-18"))
    for figure in ("learning_curves.png", "examples.png", "misclassifications.png"):
        path = RUN_DIR / name / figure
        if path.exists():
            display(DisplayImage(filename=str(path)))
    note = RUN_DIR / name / "misclassifications.txt"
    if note.exists():
        print(note.read_text())

## Inference on a known validation image
Use the trained pretrained checkpoint and saved class mapping. Show the true label alongside the prediction. Confidence is a softmax score, **not** a calibrated probability or evidence of reliability on field photos. You can replace the image with your own PIL image or file path.

In [ ]:
import csv
from PIL import Image as PILImage
from src.predict import Predictor

predictor = Predictor.from_checkpoint(RUN_DIR / "pretrained/best.pt", device="cuda")
with (RUN_DIR / "pretrained/validation_predictions.csv").open() as handle:
    example = next(csv.DictReader(handle))
image_path = CACHE_DIR / "images" / example["image_path"]
with PILImage.open(image_path) as source:
    leaf = source.convert("RGB")
display(leaf)
print("Actual:", example["actual"])
print("Prediction:", predictor.predict(leaf))

## Persist and report
Drive mode has already saved results persistently. Optionally download a ZIP for the demo; it includes the two checkpoints, so it can be around 100 MB. Do not commit this ZIP, datasets or checkpoints.

For the progress report, record: GPU/environment, source commit and dataset revision, actual training/validation counts, both validation accuracy and macro F1 scores, selected epochs, learning curves, representative predictions and errors. Discuss the direction and size of the difference only after observing it. Later repeat with fractions 0.01, 0.03 and 0.30 and multiple seeds; keep validation fixed for each seed. Classifier placement is deferred.

In [ ]:
# Optional download when using ephemeral storage, or for a separate demo backup.
DOWNLOAD_ZIP = False
if DOWNLOAD_ZIP:
    import shutil
    from google.colab import files
    archive = shutil.make_archive(str(Path(tempfile.gettempdir()) / RUN_DIR.name), "zip", RUN_DIR)
    files.download(archive)
print("Saved experiment:", RUN_DIR)
print("Comparison table:", RUN_DIR / "comparison.csv")